In [2]:
import requests
import random
import os
import json
import csv
import logging
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

logging.basicConfig(level=logging.INFO)

# Wikidata

In [3]:
# Decreasing the size of the Wikidata5M dataset

file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_30k.tsv'
num_samples = 40_000

with open(file, 'r') as f:
    lines = f.readlines()

random_sample = random.sample(lines, num_samples)

with open(output_file, 'w') as f:
    f.writelines(random_sample)

## Exploratory Analysis

In [21]:
dataset = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

df = pd.read_csv(dataset, sep='\t', header=None)
df.columns = ['head', 'relation', 'tail']
num_relations = df['relation'].unique()
print(f"Number of relations: {len(num_relations)}")

relations_counts = df['relation'].value_counts()
print(f"Value counts of relation: {relations_counts}")

# choosing relations with more than 50 entities 
relations = relations_counts[relations_counts > 50].index.tolist()
print(f"Number of relations after filtering: {len(relations)}")

Number of relations: 50
Value counts of relation: relation
P31      8392
P17      3005
P27      2507
P106     2406
P131     2005
P54      2005
P19      1867
P735     1832
P161     1113
P641     1062
P69       960
P47       906
P421      882
P105      817
P136      806
P171      750
P20       609
P495      569
P1412     521
P1344     486
P166      398
P175      396
P413      396
P264      334
P155      320
P156      305
P364      297
P361      295
P102      276
P734      235
P150      235
P57       215
P463      210
P279      208
P407      205
P159      180
P108      175
P39       165
P3373     164
P937      156
P607      153
P360      144
P527      143
P40       140
P86       136
P162      128
P50       125
P137      122
P141      113
P22       108
Name: count, dtype: int64
Number of relations after filtering: 50


In [24]:
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

# Initialize a dictionary to count triples for each relation
relation_counts = defaultdict(int)

# Load and analyze the dataset
with open(file_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            # Assuming the format is: head_entity, relation, tail_entity
            relation = parts[1]
            relation_counts[relation] += 1

# Sort the relations by their counts in descending order
sorted_relations = sorted(relation_counts.items(), key=lambda item: item[1], reverse=True)

# Print the total number of relations and some examples of counts
print(f"Total number of unique relations: {len(relation_counts)}")
for relation, count in sorted_relations[:10]:
    print(f"Relation: {relation}, Count: {count}")

Total number of unique relations: 200
Relation: P31, Count: 7547
Relation: P17, Count: 2702
Relation: P27, Count: 2254
Relation: P106, Count: 2164
Relation: P131, Count: 1803
Relation: P54, Count: 1803
Relation: P19, Count: 1679
Relation: P735, Count: 1648
Relation: P161, Count: 1001
Relation: P641, Count: 955


## Reducing the dataset size through proportional sampling

In [3]:
def proportional_sample(file_path, num_samples, target_num_relations, min_samples_per_relation=10):
    relation_counts = defaultdict(int)
    relation_triples = defaultdict(list)

    # First pass: count occurrences and collect triples
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                relation = parts[1]
                relation_counts[relation] += 1
                relation_triples[relation].append(line)
    
    # Select the top N relations by count
    top_relations = sorted(relation_counts, key=relation_counts.get, reverse=True)[:target_num_relations]

    # Adjusted total count to only consider top relations
    total_count = sum(relation_counts[relation] for relation in top_relations)
    
    # Proportional sampling within the top relations
    sampled_triples = []
    for relation in top_relations:
        proportion = relation_counts[relation] / total_count
        samples_for_relation = max(int(proportion * num_samples), min_samples_per_relation)
        
        # Ensure not to exceed the actual number of available triples
        samples_for_relation = min(samples_for_relation, len(relation_triples[relation]))
        
        sampled_triples.extend(random.sample(relation_triples[relation], samples_for_relation))
    
    return sampled_triples

def write_to_file(output_file, sampled_triples):
    with open(output_file, 'w') as f:
        f.writelines(sampled_triples)

# Example usage parameters
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'  # Update this to your actual file path
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k2.tsv'   # Desired output file path
num_samples = 40000  # Target number of samples in the reduced dataset
target_num_relations = 200  # Target number of relations to keep
min_samples_per_relation = 50  # Minimum samples per relation to ensure representation

# Execute the sampling
sampled_triples = proportional_sample(file_path, num_samples, target_num_relations, min_samples_per_relation)
write_to_file(output_file, sampled_triples)


## Wikipedia Descriptions

TODO:

1. Delete all entities that are not in the final dataset

2. Take only the first 2 sentences for each entity

In [3]:
# 1. Delete all entities that are not in the final dataset

def filter_descriptions(triples_path, descriptions_path, output_path):
    # Step 1: Read the Wikidata triples and extract unique entity IDs
    entity_ids = set()
    with open(triples_path, 'r') as triples_file:
        for line in triples_file:
            parts = line.strip().split('\t')
            if len(parts) >= 3:  # Ensure the line is valid
                entity_ids.add(parts[0])  # Add subject entity ID

    # Step 2: Filter the Wikipedia descriptions
    filtered_descriptions = []
    with open(descriptions_path, 'r') as descriptions_file:
        for line in descriptions_file:
            entity_id = line.strip().split('\t')[0]
            if entity_id in entity_ids:
                filtered_descriptions.append(line)

    # Step 3: Save the filtered descriptions to a new file
    with open(output_path, 'w') as output_file:
        for description in filtered_descriptions:
            output_file.write(description)

    print(f"Filtered descriptions saved to {output_path}")

triples_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv' 
descriptions_path = '/Users/giacomomunda/Downloads/wikidata5m_text.txt'
output_path = './../../data/wikidata5m_descriptions/wikidata5m_text_filtered.tsv'

filter_descriptions(triples_path, descriptions_path, output_path)


Filtered descriptions saved to ./../../data/wikidata5m_descriptions/wikidata5m_text_filtered.tsv


In [2]:
entity_ids = set()
with open(triples_path, 'r') as triples_file:
    for line in triples_file:
        parts = line.strip().split('\t')
        if len(parts) >= 3:  # Ensure the line is valid
            entity_ids.add(parts[0])  # Add subject entity ID

print(f"Number of unique entities: {len(entity_ids)}")

Number of unique entities: 43525
